In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time

In [100]:
import torch


class ForwardSDE:
    def __init__(self, config):
        self.config = config

    def forward(self, x):

        dt = self.config.step_size
        if self.config.method == "smld":  # SMLD (VE SDE)
            for i in range(1, self.config.max_steps):
                noise = torch.randn_like(x)
                sigma_diff = torch.sqrt(self.config.sigmas[i]**2 - self.config.sigmas[i-1]**2)
                x = x + noise * sigma_diff

        elif self.config.method == "ddim":  # DDIM (VP SDE)
            for i in range(1, self.config.max_steps):
                noise = torch.randn_like(x)
                # Euler–Maruyama discretization of: dx = -0.5 * beta(t) * x dt + sqrt(beta(t)) dW
                drift = -0.5 * self.config.betas[i] * x * dt
                diffusion = torch.sqrt(self.config.betas[i]) * noise
                x = x + drift + diffusion

        elif self.config.method == "subvp":  # Sub-VP SDE
            # precompute an approximation of the integral of beta(s) ds using cumulative sum
            # this gives a tensor cum_beta where cum_beta[i] approximates ∫₀^(tᵢ) beta(s) ds.
            cum_beta = torch.cumsum(self.config.betas, dim=0) * dt
            for i in range(1, self.config.max_steps):
                noise = torch.randn_like(x)
                # diffusion term: sqrt(beta(t) * (1 - exp(-2∫₀^(t) beta(s) ds)))
                diffusion = torch.sqrt(self.config.betas[i] * (1 - torch.exp(-2 * cum_beta[i])))
                drift = -0.5 * self.config.betas[i] * x * dt
                x = x + drift + diffusion * noise

        else:
            raise ValueError(f"unknown method: {self.config.method}")
            
        return x
    

class Config:
    def __init__(self, method=None, start=None, end=None, max_steps=None, sigma_min=None, sigma_max=None, beta_range=None):
        self.method = method or "smld"
        self.start = start or 0
        self.end = end or 1
        self.max_steps = max_steps or 1000
        self.step_size = (self.end - self.start) / self.max_steps  
        self.sigma_min = sigma_min or 0.01
        self.sigma_max = sigma_max or 1.0
        self.beta_range = beta_range or (1e-4, 0.02)
        
        # compute hyperparameters
        self.sigmas, self.betas, self.t = self.compute_params()

    def compute_params(self):
        t = torch.linspace(self.start, self.end, self.max_steps)  # time steps
        sigmas = self.sigma_min * (self.sigma_max / self.sigma_min) ** t  # geometric variance schedule
        betas = torch.linspace(self.beta_range[0], self.beta_range[1], self.max_steps)
        return sigmas, betas, t
    
    





    


In [104]:

x = torch.zeros(100, 100)
x[30:70, 30:70] =  1.0
conf = Config(method="smfld")
f = ForwardSDE(conf)
g = f.forward(x)
sns.heatmap(x)
plt.show()
sns.heatmap(g)
plt.show()
# for i in range(10):
#     plt.plot(m[i, :])
# plt.show()

ValueError: unknown method: smfld

In [83]:
x = torch.randn(100, 100) 
print(torch.var(x))

tensor(1.0288)


In [47]:
a = torch.tensor([1,2,3,4,5])
b = torch.tensor([2,2,2,2,2])
print(a**b)

tensor([ 1,  4,  9, 16, 25])


In [59]:

betas =  0.01 * (0.2/0.01)**torch.linspace(1, 10, 1000)
print(betas.shape)
print(betas[999])

torch.Size([1000])
tensor(1.0240e+11)


In [2]:
for i in reversed(range(1, 10)):
    print(i)

9
8
7
6
5
4
3
2
1
